# Chapter 1 to Linear Models: Practice Notebook
*Introduction to Machine Learning with Python (Müller & Guido), pages 1 to 68*

**How to use this:** read the explanation, run the cell, then try retyping it from memory in a blank cell underneath. Change numbers (k, alpha, C) and predict what will happen *before* you run it.

Every model follows the same workflow:
1. Load data
2. Split into train / test
3. Create the model (set its parameters)
4. `fit` on training data
5. `predict` on new data
6. `score` on test data
7. Tune the complexity knob and compare train vs test

## Setup
Run this once. If `mglearn` is missing, run `pip install mglearn` in a terminal (or `!pip install mglearn` in a cell).

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import mglearn
import sklearn

from sklearn.model_selection import train_test_split

print("scikit-learn version:", sklearn.__version__)

---
# Chapter 1

## Ex 1: Explore the Iris dataset

`load_iris()` returns a **Bunch**, which behaves like a Python dictionary. The important keys:
- `data`: the features, a 2D array of shape (150, 4). Rows are flowers, columns are measurements.
- `target`: the labels, a 1D array of 150 numbers (0, 1, 2).
- `target_names`: what 0, 1, 2 mean (setosa, versicolor, virginica).
- `feature_names`: what the 4 columns are.

Always look at your data before modeling.

In [ ]:
from sklearn.datasets import load_iris

iris_dataset = load_iris()

print("Keys:", iris_dataset.keys())
print("Feature names:", iris_dataset['feature_names'])
print("Target names:", iris_dataset['target_names'])
print("Shape of data:", iris_dataset['data'].shape)    # (samples, features)
print("Shape of target:", iris_dataset['target'].shape)
print("First 5 rows:\n", iris_dataset['data'][:5])
print("Target:\n", iris_dataset['target'])

**Notice:** the target is sorted (all 0s, then 1s, then 2s). That's why `train_test_split` **shuffles** before splitting. Otherwise your test set would be only class 2.

## Ex 1b: Look at the data with a pair plot

A scatter matrix plots every feature against every other feature, colored by class. If the classes form separate clusters, a model will probably do well.

`pd.plotting.scatter_matrix` needs a pandas DataFrame, so we convert first.

In [ ]:
import pandas as pd

X_train, X_test, y_train, y_test = train_test_split(
    iris_dataset['data'], iris_dataset['target'], random_state=0)

iris_df = pd.DataFrame(X_train, columns=iris_dataset.feature_names)
pd.plotting.scatter_matrix(iris_df, c=y_train, figsize=(12, 12), marker='o',
                           hist_kwds={'bins': 20}, s=40, alpha=.8, cmap=mglearn.cm3)
plt.show()

**What to see:** the petal length and petal width plots separate the 3 colors cleanly. That's a good sign.

## Ex 2: The full Iris pipeline

- `train_test_split` shuffles, then puts **75%** in train and **25%** in test by default.
- `random_state=0` fixes the shuffle so you get the same split every run.
- It returns 4 things in this exact order: `X_train, X_test, y_train, y_test`.
- Capital `X` = 2D feature matrix, lowercase `y` = 1D label vector (math convention).
- `KNeighborsClassifier(n_neighbors=1)`: predict using the single closest training point.
- Accuracy = fraction of correct predictions. `y_pred == y_test` gives True/False per sample, and the mean of True/False is the fraction of Trues.

In [ ]:
from sklearn.neighbors import KNeighborsClassifier

X_train, X_test, y_train, y_test = train_test_split(
    iris_dataset['data'], iris_dataset['target'], random_state=0)

print("X_train shape:", X_train.shape)   # (112, 4)
print("X_test shape:", X_test.shape)     # (38, 4)

knn = KNeighborsClassifier(n_neighbors=1)
knn.fit(X_train, y_train)                # for kNN, "fit" just stores the data

y_pred = knn.predict(X_test)
print("Test predictions:", y_pred)

# Two ways to get accuracy
print("Accuracy (np.mean):  {:.2f}".format(np.mean(y_pred == y_test)))
print("Accuracy (.score):   {:.2f}".format(knn.score(X_test, y_test)))

**Expected:** 0.97, so 37 of 38 test flowers are correct.

## Ex 3: Predict one new flower

**The #1 beginner error:** scikit-learn always wants a **2D** input (samples × features). One flower is 1 sample × 4 features, so shape `(1, 4)`. That's why there are **double brackets**.

`predict` returns a number (like `[0]`). Use it as an index into `target_names` to get the species name.

In [ ]:
X_new = np.array([[5, 2.9, 1, 0.2]])
print("X_new shape:", X_new.shape)       # (1, 4), NOT (4,)

prediction = knn.predict(X_new)
print("Prediction (number):", prediction)
print("Prediction (name):", iris_dataset['target_names'][prediction])

---
# Chapter 2: k-Nearest Neighbors

## Ex 4: kNN classification on forge

`forge` is a tiny made-up dataset: 26 points, 2 features, 2 classes. 2 features means we can draw it.

With `n_neighbors=3`, each prediction is a **majority vote** of the 3 closest training points.

In [ ]:
X, y = mglearn.datasets.make_forge()
print("X.shape:", X.shape)

mglearn.discrete_scatter(X[:, 0], X[:, 1], y)
plt.legend(["Class 0", "Class 1"], loc=4)
plt.xlabel("First feature")
plt.ylabel("Second feature")
plt.show()

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

clf = KNeighborsClassifier(n_neighbors=3)
clf.fit(X_train, y_train)

print("Test set predictions:", clf.predict(X_test))
print("Test set accuracy: {:.2f}".format(clf.score(X_test, y_test)))

**Expected:** 0.86.

**Note on `X[:, 0]`:** `:` means "all rows", `0` means "column 0". So `X[:, 0]` is the first feature for every point.

## Ex 4b: Decision boundaries for k = 1, 3, 9

The **decision boundary** is where the model switches from predicting class 0 to class 1. It's built by running kNN on every point of the plane and coloring by prediction.

- `zip([1, 3, 9], axes)` pairs each k with one subplot.
- `.fit()` returns the model itself, so you can create and fit in one line.

**Watch:** k=1 is jagged (complex, overfits), k=9 is smooth (simple).

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(10, 3))

for n_neighbors, ax in zip([1, 3, 9], axes):
    clf = KNeighborsClassifier(n_neighbors=n_neighbors).fit(X, y)
    mglearn.plots.plot_2d_separator(clf, X, fill=True, eps=0.5, ax=ax, alpha=.4)
    mglearn.discrete_scatter(X[:, 0], X[:, 1], y, ax=ax)
    ax.set_title("{} neighbor(s)".format(n_neighbors))
    ax.set_xlabel("feature 0")
    ax.set_ylabel("feature 1")

axes[0].legend(loc=3)
plt.show()

## Ex 5: Complexity curve on breast cancer

This is **the most important kNN cell**. It shows overfitting and underfitting with real data.

- Real dataset: 569 tumors, 30 features, benign vs malignant.
- `stratify=cancer.target` keeps the same benign/malignant ratio in train and test.
- For each k from 1 to 10, record **both** train accuracy and test accuracy, then plot both.

**How to read the plot:**
- k=1: train = 100% (each training point is its own nearest neighbor), test lower → **overfitting**
- k=10: both drop → **underfitting**
- Best test score around k=6 → **sweet spot**

In [ ]:
from sklearn.datasets import load_breast_cancer

cancer = load_breast_cancer()
print("Data shape:", cancer.data.shape)

X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=66)

training_accuracy = []
test_accuracy = []
neighbors_settings = range(1, 11)        # 1, 2, ..., 10

for n_neighbors in neighbors_settings:
    clf = KNeighborsClassifier(n_neighbors=n_neighbors)
    clf.fit(X_train, y_train)
    training_accuracy.append(clf.score(X_train, y_train))
    test_accuracy.append(clf.score(X_test, y_test))

plt.plot(neighbors_settings, training_accuracy, label="training accuracy")
plt.plot(neighbors_settings, test_accuracy, label="test accuracy")
plt.ylabel("Accuracy")
plt.xlabel("n_neighbors")
plt.legend()
plt.show()

best_k = neighbors_settings[np.argmax(test_accuracy)]
print("Best k on test set:", best_k)

**Remember:** for kNN, **small k = complex**, **large k = simple**. It's backwards compared to most models.

## Ex 6: kNN regression on wave

Same idea, but for predicting a **number**:
- Classification: **vote** among the k neighbors.
- Regression: **average** the k neighbors' values.

`wave` has 1 feature and a continuous target.

For regressors, `.score()` returns **R²** (not accuracy):
- R² = 1: perfect predictions
- R² = 0: no better than always predicting the mean of `y_train`
- R² < 0: worse than predicting the mean

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

X, y = mglearn.datasets.make_wave(n_samples=40)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

reg = KNeighborsRegressor(n_neighbors=3)
reg.fit(X_train, y_train)

print("Test set predictions:\n", reg.predict(X_test))
print("Test set R^2: {:.2f}".format(reg.score(X_test, y_test)))

**Expected:** R² = 0.83.

## Ex 6b: See kNN regression for k = 1, 3, 9

- `np.linspace(-3, 3, 1000)` makes 1000 evenly spaced numbers.
- `.reshape(-1, 1)` turns them into a column (1000 rows × 1 feature), because sklearn wants 2D. The `-1` means "figure out this number for me".

**Watch:** k=1 passes through every training point (jumpy, overfits). k=9 is smooth but misses detail.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
line = np.linspace(-3, 3, 1000).reshape(-1, 1)

for n_neighbors, ax in zip([1, 3, 9], axes):
    reg = KNeighborsRegressor(n_neighbors=n_neighbors)
    reg.fit(X_train, y_train)
    ax.plot(line, reg.predict(line))
    ax.plot(X_train, y_train, '^', c=mglearn.cm2(0), markersize=8)
    ax.plot(X_test, y_test, 'v', c=mglearn.cm2(1), markersize=8)
    ax.set_title("{} neighbor(s)\n train score: {:.2f} test score: {:.2f}".format(
        n_neighbors, reg.score(X_train, y_train), reg.score(X_test, y_test)))
    ax.set_xlabel("Feature")
    ax.set_ylabel("Target")

axes[0].legend(["Model predictions", "Training data/target", "Test data/target"], loc="best")
plt.show()

**kNN summary:**
- Knob: `n_neighbors` (3 or 5 is a good start). Distance: Euclidean by default.
- Good: easy to understand, quick baseline.
- Bad: slow prediction on big data, struggles with many features or sparse data, needs scaled features (Chapter 3).

---
# Chapter 2: Linear Models

## The idea
A linear model predicts with a **weighted sum** of the features:

$$\hat{y} = w[0] \cdot x[0] + w[1] \cdot x[1] + \dots + w[p] \cdot x[p] + b$$

- `w` = weights (coefficients), one per feature. In sklearn: `coef_`
- `b` = intercept (offset). In sklearn: `intercept_`
- With 1 feature it's just a line: $\hat{y} = w \cdot x + b$ (slope and intercept).

**Trailing underscore rule:** anything sklearn *learned from the training data* ends in `_` (like `coef_`). That separates it from parameters *you* set (like `alpha`).

## Ex 7: Linear regression (ordinary least squares) on wave

`LinearRegression` finds `w` and `b` that minimize the **mean squared error** between predictions and true values on the training set. It has **no parameters to tune**, which means you can't control its complexity.

In [ ]:
from sklearn.linear_model import LinearRegression

X, y = mglearn.datasets.make_wave(n_samples=60)
X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=42)

lr = LinearRegression().fit(X_train, y_train)

print("lr.coef_:", lr.coef_)            # slope, one per feature
print("lr.intercept_:", lr.intercept_)  # a single number

print("Training set score: {:.2f}".format(lr.score(X_train, y_train)))
print("Test set score: {:.2f}".format(lr.score(X_test, y_test)))

**Expected:** train 0.67, test 0.66.

**Diagnosis:** the scores are low **and** close together → **underfitting**. A straight line is too simple for 1 feature. But with **many** features, linear models become powerful and can overfit. Next cell.

## Ex 8: LinearRegression vs Ridge on extended Boston

**Extended Boston:** 506 houses, 104 features (the 13 originals plus all pairwise products of them). Lots of features → easy to overfit.

**Ridge** = linear regression + a penalty that keeps the weights **small** (close to 0, but never exactly 0). This is called **L2 regularization**. Small weights mean each feature has less influence, so the model is simpler.

**The knob is `alpha`:**
- **Higher alpha** → stronger penalty → weights pushed toward 0 → **simpler** → less overfitting
- **Lower alpha** → weaker penalty → acts more like plain LinearRegression → **more complex**

> **Note:** `load_extended_boston()` downloads the data from the internet the first time. If it fails, check your connection.

In [ ]:
from sklearn.linear_model import Ridge

X, y = mglearn.datasets.load_extended_boston()
print("X.shape:", X.shape)

X_train, X_test, y_train, y_test = train_test_split(X, y, random_state=0)

lr = LinearRegression().fit(X_train, y_train)
print("LinearRegression  train: {:.2f}  test: {:.2f}".format(
    lr.score(X_train, y_train), lr.score(X_test, y_test)))

for alpha in [0.1, 1, 10]:
    ridge = Ridge(alpha=alpha).fit(X_train, y_train)
    print("Ridge alpha={:<4}  train: {:.2f}  test: {:.2f}".format(
        alpha, ridge.score(X_train, y_train), ridge.score(X_test, y_test)))

**Book's numbers:**

| Model | Train | Test |
|---|---|---|
| LinearRegression | 0.95 | 0.61 ← overfitting |
| Ridge alpha=10 | 0.79 | 0.64 |
| Ridge alpha=1 | 0.89 | 0.75 |
| Ridge alpha=0.1 | 0.93 | 0.77 ← best |

**Pattern:** as alpha goes up, train score drops (simpler), and test score goes up then down (sweet spot).

## Ex 8b: Plot Ridge coefficients for different alphas

Each dot is one feature's weight. Higher alpha squeezes the weights toward 0.

In [ ]:
ridge10 = Ridge(alpha=10).fit(X_train, y_train)
ridge1 = Ridge(alpha=1).fit(X_train, y_train)
ridge01 = Ridge(alpha=0.1).fit(X_train, y_train)

plt.plot(ridge10.coef_, 's', label="Ridge alpha=10")
plt.plot(ridge1.coef_, '^', label="Ridge alpha=1")
plt.plot(ridge01.coef_, 'v', label="Ridge alpha=0.1")
plt.plot(lr.coef_, 'o', label="LinearRegression")
plt.xlabel("Coefficient index")
plt.ylabel("Coefficient magnitude")
plt.hlines(0, 0, len(lr.coef_))
plt.ylim(-25, 25)
plt.legend()
plt.show()

**Lesson from Figure 2-13 (learning curves):** with **enough data**, regularization matters less, and LinearRegression catches up with Ridge. More data = less overfitting.

## Ex 9: Lasso

**Lasso** = linear regression + a different penalty (**L1 regularization**) that can push weights to **exactly 0**. A weight of 0 means that feature is ignored completely. So Lasso does **automatic feature selection**.

Same knob, `alpha`, same direction: higher alpha = simpler.

- `np.sum(lasso.coef_ != 0)` counts features with a nonzero weight, i.e. features actually used.
- For small alpha, Lasso needs more iterations to converge, so set `max_iter=100000`.

In [ ]:
from sklearn.linear_model import Lasso

for alpha in [1, 0.01, 0.0001]:
    lasso = Lasso(alpha=alpha, max_iter=100000).fit(X_train, y_train)
    print("Lasso alpha={:<6}  train: {:.2f}  test: {:.2f}  features used: {}".format(
        alpha, lasso.score(X_train, y_train), lasso.score(X_test, y_test),
        np.sum(lasso.coef_ != 0)))

**Book's numbers:**

| alpha | Train | Test | Features used (of 104) |
|---|---|---|---|
| 1 | 0.29 | 0.21 | 4 ← underfitting |
| 0.01 | 0.90 | 0.77 | 33 ← best |
| 0.0001 | 0.95 | 0.64 | 94 ← overfitting |

**Ridge vs Lasso:**
- Usually try **Ridge first**.
- Use **Lasso** when you expect only a few features matter, or you want an easier-to-explain model.
- `ElasticNet` combines both penalties.

## Linear models for classification

Same weighted sum, but instead of outputting the number, check its **sign**:

$$\hat{y} = w[0] \cdot x[0] + \dots + w[p] \cdot x[p] + b > 0$$

If > 0, predict class 1, otherwise class 0. The decision boundary is a **straight line** (in 2D), a plane in 3D, and so on.

Two common ones:
- `LogisticRegression` (a **classifier** despite the name)
- `LinearSVC` (linear support vector classifier)

**The knob is `C`, and it runs the OPPOSITE way from alpha:**
- **Higher C** → **less** regularization → fits the training data harder → **more complex**
- **Lower C** → **more** regularization → **simpler**

Memory trick: **alpha up = simpler. C up = more complex.**

## Ex 10a: Decision boundaries on forge

In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.svm import LinearSVC

X, y = mglearn.datasets.make_forge()

fig, axes = plt.subplots(1, 2, figsize=(10, 3))
for model, ax in zip([LinearSVC(), LogisticRegression()], axes):
    clf = model.fit(X, y)
    mglearn.plots.plot_2d_separator(clf, X, fill=False, eps=0.5, ax=ax, alpha=.7)
    mglearn.discrete_scatter(X[:, 0], X[:, 1], y, ax=ax)
    ax.set_title(clf.__class__.__name__)
    ax.set_xlabel("Feature 0")
    ax.set_ylabel("Feature 1")
axes[0].legend()
plt.show()

# How C changes the boundary
mglearn.plots.plot_linear_svc_regularization()
plt.show()

**Watch the second plot:** small C gives a nearly flat line that misses 2 points. Large C tilts the line hard to get as many training points right as possible.

## Ex 10b: LogisticRegression with C on breast cancer

`max_iter=10000` gives the solver enough steps to converge (avoids a warning; the book's older sklearn didn't need it).

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42)

models = {}
for C in [0.01, 1, 100]:
    logreg = LogisticRegression(C=C, max_iter=10000).fit(X_train, y_train)
    models[C] = logreg
    print("C={:<5}  train: {:.3f}  test: {:.3f}".format(
        C, logreg.score(X_train, y_train), logreg.score(X_test, y_test)))

**Book's numbers** (newer scikit-learn gives slightly different ones, roughly 0.953/0.951, 0.958/0.958, 0.981/0.965; the pattern is the same):

| C | Train | Test |
|---|---|---|
| 0.01 | 0.934 | 0.930 ← underfitting |
| 1 | 0.953 | 0.958 |
| 100 | 0.972 | 0.965 ← best here |

At C=1, train and test are almost equal, a hint that the model is **underfitting** and could handle more complexity. So we raise C.

## Ex 10c: Plot the coefficients

`.coef_` has shape (1, 30) for binary classification. `.T` (transpose) flips it to (30, 1) so it plots as 30 points.

In [ ]:
plt.figure(figsize=(10, 5))
plt.plot(models[1].coef_.T, 'o', label="C=1")
plt.plot(models[100].coef_.T, '^', label="C=100")
plt.plot(models[0.01].coef_.T, 'v', label="C=0.01")
plt.xticks(range(cancer.data.shape[1]), cancer.feature_names, rotation=90)
plt.hlines(0, 0, cancer.data.shape[1])
plt.ylim(-5, 5)
plt.xlabel("Feature")
plt.ylabel("Coefficient magnitude")
plt.legend()
plt.show()

**Careful interpreting signs:** a feature's coefficient can flip sign between models (positive at one C, negative at another). So "positive weight = feature means malignant" isn't always a safe conclusion. Interpret coefficients with a grain of salt.

## Ex 11: L1-regularized logistic regression

`LogisticRegression` uses **L2** (like Ridge) by default. Switch to **L1** (like Lasso) to get feature selection: many coefficients become exactly 0.

**Version note:** the book writes `penalty="l1"`. In scikit-learn 1.8 and newer, that's deprecated and the new way is `l1_ratio=1`. The code below checks your version and uses the right one. L1 also needs `solver="liblinear"`.

In [ ]:
from sklearn.utils.fixes import parse_version

new_sklearn = parse_version(sklearn.__version__) >= parse_version("1.8")

plt.figure(figsize=(10, 5))
for C, marker in zip([0.001, 1, 100], ['o', '^', 'v']):
    if new_sklearn:
        lr_l1 = LogisticRegression(C=C, l1_ratio=1, solver="liblinear", max_iter=10000)
    else:
        lr_l1 = LogisticRegression(C=C, penalty="l1", solver="liblinear", max_iter=10000)
    lr_l1.fit(X_train, y_train)

    print("C={:<6}  train: {:.2f}  test: {:.2f}  features used: {}".format(
        C, lr_l1.score(X_train, y_train), lr_l1.score(X_test, y_test),
        np.sum(lr_l1.coef_ != 0)))
    plt.plot(lr_l1.coef_.T, marker, label="C={:.3f}".format(C))

plt.xticks(range(cancer.data.shape[1]), cancer.feature_names, rotation=90)
plt.hlines(0, 0, cancer.data.shape[1])
plt.ylim(-5, 5)
plt.xlabel("Feature")
plt.ylabel("Coefficient magnitude")
plt.legend(loc=3)
plt.show()

**Book's numbers:** C=0.001 → 0.91 / 0.92, C=1 → 0.96 / 0.96, C=100 → 0.99 / 0.98. Small C uses only a few features.

## Ex 12: Multiclass with one-vs-rest

Most linear classifiers are naturally **binary**. For 3 classes, **one-vs-rest** trains **3 binary classifiers**:
- Class 0 vs everything else
- Class 1 vs everything else
- Class 2 vs everything else

To predict, run all 3 and pick the class whose classifier gives the **highest score**.

So `coef_` has shape **(3, 2)**: 3 classifiers (rows) × 2 features (columns). `intercept_` has shape **(3,)**: one intercept per classifier.

In [ ]:
from sklearn.datasets import make_blobs

X, y = make_blobs(random_state=42)

linear_svm = LinearSVC().fit(X, y)
print("Coefficient shape:", linear_svm.coef_.shape)    # (3, 2)
print("Intercept shape:", linear_svm.intercept_.shape) # (3,)

# Draw each classifier's line and the full decision regions
mglearn.plots.plot_2d_classification(linear_svm, X, fill=True, alpha=.7)
mglearn.discrete_scatter(X[:, 0], X[:, 1], y)

line = np.linspace(-15, 15)
for coef, intercept, color in zip(linear_svm.coef_, linear_svm.intercept_, ['b', 'r', 'g']):
    # boundary: coef[0]*x0 + coef[1]*x1 + intercept = 0  ->  solve for x1
    plt.plot(line, -(line * coef[0] + intercept) / coef[1], c=color)

plt.ylim(-10, 15)
plt.xlim(-10, 8)
plt.xlabel("Feature 0")
plt.ylabel("Feature 1")
plt.legend(['Class 0', 'Class 1', 'Class 2', 'Line class 0', 'Line class 1', 'Line class 2'],
           loc=(1.01, 0.3))
plt.show()

**Where the line formula comes from (pure algebra):** the boundary is where the score is 0:

$$w_0 x_0 + w_1 x_1 + b = 0 \;\Rightarrow\; x_1 = -\frac{w_0 x_0 + b}{w_1}$$

That's exactly `-(line * coef[0] + intercept) / coef[1]`.

## Bonus: method chaining

`.fit()` returns the model itself, so you can chain calls:

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    cancer.data, cancer.target, stratify=cancer.target, random_state=42)

# create + fit in one line (common, readable)
logreg = LogisticRegression(max_iter=10000).fit(X_train, y_train)

# fit + predict in one line
y_pred = LogisticRegression(max_iter=10000).fit(X_train, y_train).predict(X_test)
print(y_pred[:10])

The all-in-one version works but you lose the fitted model (no variable), so you can't inspect `coef_` or reuse it.

---
# Cheat sheet

| Model | Task | Complexity knob | Higher knob means |
|---|---|---|---|
| `KNeighborsClassifier` / `Regressor` | both | `n_neighbors` | **simpler** |
| `LinearRegression` | regression | none | n/a |
| `Ridge` (L2) | regression | `alpha` | **simpler** |
| `Lasso` (L1) | regression | `alpha` | **simpler** (+ more weights = 0) |
| `LogisticRegression` | classification | `C` | **more complex** |
| `LinearSVC` | classification | `C` | **more complex** |

**Diagnosing:**
- Train ≫ test → overfitting → make it simpler
- Train ≈ test, both low → underfitting → make it more complex

**When to use linear models:** very fast to train and predict, scale to huge datasets, work well when there are many features (even more features than samples), and work well with sparse data. Weakness: in low dimensions, a straight line may be too simple.